In [31]:
path1 = '/code/jixurui/data/LLM_chuilei/data/train_data/train1.json'
path1 = '/code/jixurui/data/LLM_chuilei/data/process_data/new_data_sorce_category_label_cursor_postprocess_v3_high_scores_意图_cluster合并_>=0.6/意图识别_train_all/train6_domain_120w.json'
path2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/IF_188k.jsonl'
path3 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10/Instruction_Following_&_Text_Processing.jsonl'
path4 = '/workspace/data/data2136/zxd/58_base_data_over_0428_drop_duplicates/Train_Test/train_split.json'

In [32]:
import json 

def read_datas(path):
    datas = []
    try:
        with open(path, 'r', encoding='utf8') as f:
            datas = json.load(f)
    except Exception as e:
        with open(path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                datas.append(cur_)
# print(len(datas), path)
    return datas


In [33]:
data1 = read_datas(path1)

In [34]:
data2 = read_datas(path2)
data3 = read_datas(path3)

In [35]:
len(data1), len(data2), len(data3)

(1200000, 167338, 21110)

In [36]:
data1[0]

{'instruction': '角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指导原则：\n---当用户有辱骂情绪，表现为直接骂人说脏话，emotion设置为0，一般包含关键词如：尼玛、垃圾、mlgb、滚、是个屁等，用户仅表达不XX没有骂人的话，不归为该类别；案例：为什么别人虚拟联系我现实诈骗电话->无情感而非辱骂，数次好几个i滚滚滚->辱骂。\n---当用户有急切情绪，表现为嫌弃客服回复慢响应慢，单纯询问时间的不算，emotion设置为1，一般包含关键词如：我很急、着急死了、慢死了、太慢了、效率低等；案例：买会员了什么时候放款->无情感而非急切(用户只是询问某项服务具体时间)，请问排队需要多久时间->急切（用户等待客服服务已经很久）。\n---当用户有失望情绪（不满意、效果差、无体验、感觉麻烦、泄露信息、被骚扰），emotion设置为2，一般包含关键词如：很不满意、太差了、无语、毫无体验、一点用都没有、没效果、不人性化、真麻烦、号码泄露、被骚扰等；案例：VIP没用可以退吗->无情感而非失望（用户仅询问能否退款）。\n---当用户有生气情绪（说骗人、虚假信息、问题未解决、生气），必须表达气愤的情绪，只是反问的话不算，emotion设置为3，一般包含关键词如：卸载、拉黑、瞎搞、骗钱、啥都没解决、坑人、啥都是假的、气死我了、什么玩意、瞎搞、没人管、听不懂人话、太不靠谱了、再也不用了、诈骗等；案例：你不批就拒绝我谢谢 还了你这点钱这辈子不用你们这个平台->生气（很生气说不用平台了）。\n---当用户有投诉情绪，emotion设置为4，一般包含关键词如：我要投诉、12315、315投诉、报警、110等；案例：会员够卖后不成功能退款不->无情感而非投诉（用户仅询问退款问题）。\n---用户无上述情绪指向时，或者用户平静询问问题无激动情绪时，emotion设置为-1；案例：58逾期几天，上催收->无情感（用户只是平静表达情况）。\n限制：\n1、当同时存在多种情绪时，按优先级4、0、3、2、1、-1识别，只输出优先级最高的emotion结果。\n2、不要过度揣测用户意愿，需要根据指导原则给出简短合理解释，放入reason字段。\n3、对异常情绪（emotion为0、1、2、3、4）的判断不能宽泛，需要很严格。\n4、请以json字符

In [37]:
deal1 = []
for elem in data1:
    deal1.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})
len(deal1)

1200000

In [38]:
data_new = read_datas(path4)
deal_new = []
for elem in data_new:
    deal_new.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})
len(deal_new)

298989

In [39]:
deal2 = []
for elem in data2:
    deal2.append({'instruction':elem['conversations'][0]['content'], 'input': '', 'output': elem['conversations'][1]['content']})

In [40]:
deal3 = []
for elem in data3:
    deal3.append({'instruction': elem['prompt'], 'input': '', 'output': elem['output']})

In [41]:
totals = deal1 + deal2 +deal3

In [42]:
len(totals)

1388448

In [43]:
zh_110 = []
with open('/code/zhaoxudong03/qwen3_235b_2507_distill_110k.jsonl', 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        sample = {'instruction': cur_['messages'][0]['content'], 'input': '', 'output': cur_['messages'][1]['content']}
        zh_110.append(sample)

In [44]:
totals = deal1 + deal2 +deal3 + zh_110 + deal_new

In [45]:
len(totals)

1797437

In [46]:
with open('/code/jixurui/data/LLM_chuilei/data/train_data/train6_category_180w.json', 'w', encoding='utf8') as fsave:
    json.dump(totals, fsave, ensure_ascii=False, indent=4)